In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

ratings_data = pd.read_csv('../data_clean/clean_masterdata.csv',usecols=['userId', 'movieId', 'rating', 'title'])
ratings_data = ratings_data.drop_duplicates(subset=['userId', 'movieId'])
ratings_data['user_cat'] = ratings_data['userId'].astype('category')
ratings_data['movie_cat'] = ratings_data['title'].astype('category')

sparse_matrix = csr_matrix((ratings_data['rating'], (ratings_data['movie_cat'].cat.codes, ratings_data['user_cat'].cat.codes))) #csr format(data,(rosw,columns))
#rows(unique movies) cols(unique users)
print(f"User/movie grid shape: {sparse_matrix.shape}")

User/movie grid shape: (33757, 63562)


In [5]:
#The svd 

svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(sparse_matrix)

#final shape: (33757,50)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

svd_cosine_sim = cosine_similarity(latent_matrix)

print(f"Final SVD Grid Shape: {svd_cosine_sim.shape}")

Final SVD Grid Shape: (33757, 33757)


In [11]:
#the search

movie_mapping = ratings_data[['title', 'movie_cat']].drop_duplicates()
movie_mapping['row_num'] = movie_mapping['movie_cat'].cat.codes

indices = pd.Series(movie_mapping['row_num'].values, index=movie_mapping['title'].str.lower().str.strip())
indices = indices[~indices.index.duplicated(keep='first')]

def get_svd_recommendations(title, cosine_sim=svd_cosine_sim):
    clean_title = title.lower().strip()
    
    if clean_title not in indices:
        return f"'{title}' is not in the social database."
        
    idx = indices[clean_title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_10 = sim_scores[1:11]
    
    movie_indices = [i[0] for i in top_10]
    
    reverse_mapping = movie_mapping.set_index('row_num')['title']
    recommendations = reverse_mapping.loc[movie_indices].tolist()
    for i, movie in enumerate(recommendations, 1):
        print(f"{i}. {movie}")


In [12]:
get_svd_recommendations('the dark knight')

1. Inception
2. Iron Man
3. The Prestige
4. The Dark Knight Rises
5. Batman Begins
6. Inglourious Basterds
7. V for Vendetta
8. Avatar
9. The Departed
10. WALL·E
